# EOV-MMP pilot — Colab runner

**Runtime → Change runtime type → T4 GPU.** Add `HF_TOKEN` under the 🔑 icon
with notebook access on.

`Run all` is the intended way to use this. It is re-runnable: the run resumes
from Drive, so if the session drops just run it again.

Settings were decided by measurement — see `pilot_analysis/PILOT-STATUS.md`
§B.20 (frame_stride 1) and §B.21 (`--fuse-splits`). Roughly 5 h for 200 videos.


## 1. Setup


In [ ]:
!nvidia-smi -L
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
cc = torch.cuda.get_device_capability()
ARCH = f'{cc[0]}.{cc[1]}'          # 7.5 T4 · 8.0 A100 · 8.6 3090 · 8.9 L4
gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'compute {ARCH}, VRAM {gb:.1f} GiB')
assert gb > 13, 'too little VRAM even for evaluation'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!git clone -q https://github.com/ghibli613/ov-vidvrd-lab.git || (cd ov-vidvrd-lab && git pull -q)
%cd /content/ov-vidvrd-lab
!git log --oneline -1


In [ ]:
import os
# data and output on the session disk; only results go to Drive
os.environ['VIDVRD_DATA_ROOT']   = '/content/data/vidvrd'
os.environ['VIDVRD_OUTPUT_ROOT'] = '/content/output'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!mkdir -p /content/preds/full /content/output/ckpt /content/drive/MyDrive/vidvrd/preds/full


In [ ]:
!pip install -q ftfy regex einops timm fvcore pycocotools \
                opencv-python-headless gdown huggingface_hub hf_transfer


## 2. CUDA operator

`pip install .` — not `-e` (never compiles it), not `setup.py install`
(removed in setuptools 80).


In [ ]:
%cd /content/ov-vidvrd-lab/ops
!rm -rf build *.egg-info
!TORCH_CUDA_ARCH_LIST="{ARCH}" pip install --no-build-isolation . 2>&1 | tail -20
%cd /content/ov-vidvrd-lab
import torch, MultiScaleDeformableAttention
print('operator OK')


## 3. Data and weights

Annotations and GT are rebuilt locally (~2 min). **Never add `frames` to
`--steps`** — frames stream per batch during the run.

Weights come from `MyDrive/vidvrd/ckpt` (~2.8 GB). If that stash is ever lost,
uncomment the HuggingFace line.


In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!python tools/prepare_data.py --steps anno,meta,gt


In [ ]:
import os, shutil, sys
sys.path.insert(0, '/content/ov-vidvrd-lab'); os.chdir('/content/ov-vidvrd-lab')
from utils import paths
STASH = '/content/drive/MyDrive/vidvrd/ckpt'
NEED = {'clip_L14_feat_vidvrd.pkl': (paths.META_DIR, 290.8),
        'VidVRD_ECC_test.json':      (paths.META_DIR,   7.8),
        'VidVRD_ECC_train.json':     (paths.META_DIR,  35.0),
        'AFLink_epoch20.pth':        (paths.CKPT_DIR,   4.3),
        'baseline_fbce_vidvrd_bs1_lr1e-05_dim512_none_rel_mot_clip_bbox_end2end_base-001.pth': (paths.CKPT_DIR, 2579.5)}
missing = []
for name, (dest, mb) in NEED.items():
    os.makedirs(dest, exist_ok=True)
    src, dst = os.path.join(STASH, name), os.path.join(dest, name)
    if not os.path.exists(src):
        missing.append(name); continue
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        shutil.copy2(src, dst)
    got = os.path.getsize(dst)/1e6
    print(f"{'ok ' if abs(got-mb)<1 else 'SIZE?'} {got:8.1f} MB  {name[:48]}")
assert not missing, f'not in {STASH}: {missing}'

# fallback if the Drive stash is gone:
# !python tools/hugging_download.py --manifest https://huggingface.co/ghibli613/ov-vidvrd-weights/resolve/main/MANIFEST.json --only eval


## 4. The run

Resumes from Drive and skips finished batches before downloading them.
Re-run this section after any disconnect.


In [ ]:
import json, os
!cp -n /content/drive/MyDrive/vidvrd/preds/full/*.json /content/preds/full/ 2>/dev/null || true
p = '/content/preds/full/final_merged_all.json'
print(f"{len(json.load(open(p))) if os.path.exists(p) else 0}/200 videos already done")


In [ ]:
!python pilot_analysis/scripts/dump_predictions.py \
    --ckpt_path $VIDVRD_OUTPUT_ROOT/ckpt/baseline_fbce_vidvrd_bs1_lr1e-05_dim512_none_rel_mot_clip_bbox_end2end_base-001.pth \
    --path_AFLink $VIDVRD_OUTPUT_ROOT/ckpt/AFLink_epoch20.pth \
    --shards https://huggingface.co/datasets/ghibli613/ov-vidvrd-frames/resolve/main/SHARDS.json \
    --out /content/preds/full \
    --drive-copy /content/drive/MyDrive/vidvrd/preds/full \
    --disk-budget 1.0 --flush-every 2 \
    --frame_stride 1 --fuse-splits


## 5. Results

Phases 2 and 3 need no GPU — they can run on a CPU runtime or your own
machine once the dumps are on Drive.


In [ ]:
import json, os
L = '/content/preds/full'
for s in ('all','novel'):
    p = os.path.join(L, f'final_merged_{s}.json')
    n = len(json.load(open(p))) if os.path.exists(p) else 0
    print(f'{s:6} {n:3d}/200')
mp = os.path.join(L,'metrics.json')
if os.path.exists(mp):
    m = json.load(open(mp))
    for s in ('all','novel'):
        d = m.get(s, {})
        if 'mAP' in d:
            print(f"{s:6} mAP {d['mAP']*100:6.2f}  R@50 {d['R@50']*100:6.2f}  "
                  f"R@100 {d['R@100']*100:6.2f}  ({d.get('scored_over','?')})")
    print('gate: all 26.88 / novel 15.64, full test set only')
else:
    print('run unfinished -- re-run section 4')


In [ ]:
import json, os
n = len(json.load(open('/content/preds/full/final_merged_all.json')))
if n < 200:
    print(f'{n}/200 -- finish the run first')
else:
    !cp -v /content/preds/full/*.json /content/drive/MyDrive/vidvrd/preds/full/
    !python pilot_analysis/scripts/phase2_phase3.py --preds /content/preds/full \
        2>&1 | tee /content/drive/MyDrive/vidvrd/preds/phase2_phase3_results.txt
